In [1]:
# STAGE 0 — Context init + output dirs

import os
import json

ctx = {}

OUTPUT_DIR = "../../data/processed/book1/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
# STAGE 0 — Book 1 profile + configuration

ctx["profile"] = {
    "book_number": 1,
    "pdf_path": "../../data/raw/EMV_v4.4_Book_1_ICC_to_Terminal_Interface.pdf",
    "version": "4.4",
    "publication_date": "2022-10",
    "toc_pages": {
        "start": 4,
        "end": 8,
    },
    "heading_patterns": {
        "part": r"^Part\s+[IVX]+",
        "level_2": r"^\d+\s+",
        "level_3": r"^\d+\.\d+\s+",
        "level_4": r"^\d+\.\d+\.\d+\s+",
        "appendix": r"^(Annex|Appendix)\s+[A-Z]",
    },
    "noise_detection": {
        "header_zone_ratio": 0.08,
        "footer_zone_ratio": 0.08,
        "min_repetition_threshold": 5,
    },
    "output_dir": OUTPUT_DIR,
}

In [3]:
# STAGE 0 — Save profile.json

profile_path = os.path.join(ctx["profile"]["output_dir"], "profile.json")
with open(profile_path, "w", encoding="utf-8") as f:
    json.dump(ctx["profile"], f, ensure_ascii=False, indent=2)

In [4]:
# STAGE 0 — Validation

print("ctx['profile']:\n")
print(json.dumps(ctx["profile"], ensure_ascii=False, indent=2))

print("\nDirectory exists:", os.path.isdir(ctx["profile"]["output_dir"]))
print("profile.json exists:", os.path.isfile(profile_path))
print("profile.json path:", profile_path)

with open(profile_path, "r", encoding="utf-8") as f:
    loaded = json.load(f)
print("\nLoaded profile keys:", sorted(list(loaded.keys())))

ctx['profile']:

{
  "book_number": 1,
  "pdf_path": "../../data/raw/EMV_v4.4_Book_1_ICC_to_Terminal_Interface.pdf",
  "version": "4.4",
  "publication_date": "2022-10",
  "toc_pages": {
    "start": 4,
    "end": 8
  },
  "heading_patterns": {
    "part": "^Part\\s+[IVX]+",
    "level_2": "^\\d+\\s+",
    "level_3": "^\\d+\\.\\d+\\s+",
    "level_4": "^\\d+\\.\\d+\\.\\d+\\s+",
    "appendix": "^(Annex|Appendix)\\s+[A-Z]"
  },
  "noise_detection": {
    "header_zone_ratio": 0.08,
    "footer_zone_ratio": 0.08,
    "min_repetition_threshold": 5
  },
  "output_dir": "../../data/processed/book1/"
}

Directory exists: True
profile.json exists: True
profile.json path: ../../data/processed/book1/profile.json

Loaded profile keys: ['book_number', 'heading_patterns', 'noise_detection', 'output_dir', 'pdf_path', 'publication_date', 'toc_pages', 'version']


In [5]:
# STAGE 1 — Imports + ctx guards + paths + parameters (NO glossary)

import os
import json
import re
from typing import List, Dict, Any, Optional

import pdfplumber
import fitz  # PyMuPDF
import pandas as pd

if "ctx" not in globals() or not isinstance(ctx, dict):
    raise RuntimeError("ctx not found. Run STAGE 0 first.")

pdf_path = ctx.get("profile", {}).get("pdf_path")
out_dir = ctx.get("profile", {}).get("output_dir")

if not pdf_path or not os.path.isfile(pdf_path):
    raise FileNotFoundError(f"PDF not found: {pdf_path}")

if not out_dir:
    raise RuntimeError("ctx['profile']['output_dir'] is missing.")
os.makedirs(out_dir, exist_ok=True)

ctx.setdefault("reports", {})
ctx["reports"].setdefault("stage1_warnings", [])
ctx["reports"].setdefault("stage1_errors", [])

PAGES_JSONL_PATH = os.path.join(out_dir, "pages_raw.jsonl")
PAGES_SUMMARY_CSV_PATH = os.path.join(out_dir, "pages_summary.csv")
STAGE1_REPORT_PATH = os.path.join(out_dir, "stage1_report.json")

# --- PARAMETERS ---
# Pages where you want to use PyMuPDF text as the primary "line text" source (multi-column pages).
# Leave empty to disable.
PYMUPDF_TEXT_PAGES = list(range(12, 34)) #+ [33]   # e.g., [12, 13, 14]

# For pdfplumber word->line grouping (still used for geometry everywhere)

print("pdf_path:", pdf_path)
print("out_dir:", out_dir)
print("PYMUPDF_TEXT_PAGES:", PYMUPDF_TEXT_PAGES)

pdf_path: ../../data/raw/EMV_v4.4_Book_1_ICC_to_Terminal_Interface.pdf
out_dir: ../../data/processed/book1/
PYMUPDF_TEXT_PAGES: [12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33]


In [6]:
# STAGE 1 — Helpers: safe normalization + meta subset + multi-column aware words_to_lines()

def _safe_text(s):
    if s is None:
        return ""
    if not isinstance(s, str):
        s = str(s)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = "\n".join([ln.rstrip() for ln in s.split("\n")])
    return s

def _get_meta_subset(d, keys):
    meta = {}
    for k in keys:
        if k in d and d[k] is not None:
            meta[k] = d[k]
    return meta

def words_to_lines(words, y_tol=4.0, join_with=" "):
    if not words:
        return []

    ws = []
    for idx, w in enumerate(words):
        try:
            ws.append({
                "_idx": idx,  # preserve original order
                "text": _safe_text(w.get("text", "")),
                "x0": float(w.get("x0", 0.0)),
                "x1": float(w.get("x1", 0.0)),
                "top": float(w.get("top", 0.0)),
                "bottom": float(w.get("bottom", 0.0)),
                "meta": _get_meta_subset(w, ["size", "fontname", "upright", "direction", "doctop"]),
            })
        except Exception:
            continue

    if not ws:
        return []

    # IMPORTANT: no sorting here. We keep extractor order.
    row_clusters = []
    current = [ws[0]]
    current_top = ws[0]["top"]

    for w in ws[1:]:
        if abs(w["top"] - current_top) <= y_tol:
            current.append(w)
            current_top = (current_top * (len(current) - 1) + w["top"]) / len(current)
        else:
            row_clusters.append(current)
            current = [w]
            current_top = w["top"]
    row_clusters.append(current)

    # Split each row into segments by big x-gaps, but preserve the order of words as they appear
    lines = []
    for row in row_clusters:
        if not row:
            continue

        segments = []
        seg = [row[0]]
        for w in row[1:]:
            seg.append(w)
        segments.append(seg)

        for seg in segments:
            seg_text_parts = [g["text"] for g in seg if g["text"]]
            text = join_with.join(seg_text_parts).strip()
            if not text:
                continue

            x0 = min(g["x0"] for g in seg)
            x1 = max(g["x1"] for g in seg)
            top = min(g["top"] for g in seg)
            bottom = max(g["bottom"] for g in seg)

            sizes = [g["meta"].get("size") for g in seg if "size" in g["meta"]]
            meta = {}
            if sizes:
                try:
                    meta["avg_size"] = float(sum(sizes) / len(sizes))
                except Exception:
                    pass
            meta["segment_words"] = int(len(seg))

            lines.append({
                "text": text,
                "x0": x0,
                "x1": x1,
                "top": top,
                "bottom": bottom,
                "meta": meta,
            })

    return lines

In [7]:
# STAGE 1 — PyMuPDF structured lines (optional): get_text("dict") for reading-order line list

def pymupdf_lines_with_coords(pm_page) -> List[Dict[str, Any]]:
    out = []
    d = pm_page.get_text("dict")

    for b in d.get("blocks", []):
        if b.get("type") != 0:
            continue
        for ln in b.get("lines", []):
            spans = ln.get("spans", [])
            if not spans:
                continue

            texts = []
            x0s, y0s, x1s, y1s = [], [], [], []
            sizes = []
            fonts = []

            for sp in spans:
                t = _safe_text(sp.get("text", ""))
                if t:
                    texts.append(t)
                bbox = sp.get("bbox")
                if bbox and len(bbox) == 4:
                    x0, y0, x1, y1 = bbox
                    x0s.append(float(x0)); y0s.append(float(y0)); x1s.append(float(x1)); y1s.append(float(y1))
                if sp.get("size") is not None:
                    try:
                        sizes.append(float(sp["size"]))
                    except Exception:
                        pass
                if sp.get("font"):
                    fonts.append(sp["font"])

            txt = _safe_text("".join(texts)).strip()
            if not txt:
                continue

            if x0s:
                x0 = min(x0s); x1 = max(x1s); top = min(y0s); bottom = max(y1s)
            else:
                x0 = x1 = top = bottom = 0.0

            meta = {}
            if sizes:
                meta["avg_size"] = float(sum(sizes) / len(sizes))
            if fonts:
                meta["fonts_sample"] = list(dict.fromkeys(fonts))[:3]

            out.append({"text": txt, "x0": x0, "x1": x1, "top": top, "bottom": bottom, "meta": meta})

    return out

In [8]:
# STAGE 1 — Extraction: per page primitives (pdfplumber geometry + PyMuPDF text; optionally PyMuPDF lines)
# Save pages_raw.jsonl

ctx["pages"] = []

summary_rows = []
warnings = ctx["reports"]["stage1_warnings"]
errors = ctx["reports"]["stage1_errors"]

try:
    pm_doc = fitz.open(pdf_path)
except Exception as e:
    raise RuntimeError(f"PyMuPDF failed to open PDF: {e}")

try:
    pl_doc = pdfplumber.open(pdf_path)
except Exception as e:
    pm_doc.close()
    raise RuntimeError(f"pdfplumber failed to open PDF: {e}")

num_pages_pl = len(pl_doc.pages)
num_pages_pm = pm_doc.page_count
if num_pages_pl != num_pages_pm:
    warnings.append({
        "type": "page_count_mismatch",
        "pdfplumber_pages": num_pages_pl,
        "pymupdf_pages": num_pages_pm,
    })

num_pages = min(num_pages_pl, num_pages_pm)
pymupdf_text_pages_set = set(PYMUPDF_TEXT_PAGES)

with open(PAGES_JSONL_PATH, "w", encoding="utf-8") as fjsonl:
    for i in range(num_pages):
        page_num = i + 1
        page_obj = {
            "page_num": page_num,
            "width": None,
            "height": None,
            "pdfplumber": {"words": [], "lines": []},
            "pymupdf": {"text": "", "lines": None},
            "stage1": {"text_source_for_downstream": "pdfplumber"},
        }

        page_warn = []
        try:
            pl_page = pl_doc.pages[i]
            page_obj["width"] = float(getattr(pl_page, "width", None) or 0.0)
            page_obj["height"] = float(getattr(pl_page, "height", None) or 0.0)

            # pdfplumber words
            try:
                raw_words = pl_page.extract_words(
                    keep_blank_chars=False,
                    use_text_flow=True,
                    extra_attrs=["fontname", "size"],
                )
            except TypeError:
                raw_words = pl_page.extract_words()

            words = []
            for w in raw_words or []:
                try:
                    words.append({
                        "text": _safe_text(w.get("text", "")),
                        "x0": float(w.get("x0", 0.0)),
                        "x1": float(w.get("x1", 0.0)),
                        "top": float(w.get("top", 0.0)),
                        "bottom": float(w.get("bottom", 0.0)),
                        "meta": _get_meta_subset(w, ["size", "fontname", "upright", "direction", "doctop"]),
                    })
                except Exception:
                    continue
            page_obj["pdfplumber"]["words"] = words

            # pdfplumber-derived lines (geometry-first)
            page_obj["pdfplumber"]["lines"] = words_to_lines(words, y_tol=5.0)

        except Exception as e:
            page_warn.append({"type": "pdfplumber_page_error", "page_num": page_num, "error": str(e)})

        # PyMuPDF text (always)
        try:
            pm_page = pm_doc.load_page(i)
            pm_text = pm_page.get_text("text") or ""
            page_obj["pymupdf"]["text"] = _safe_text(pm_text)
        except Exception as e:
            page_warn.append({"type": "pymupdf_page_error", "page_num": page_num, "error": str(e)})
            page_obj["pymupdf"]["text"] = ""

        # If configured, also compute PyMuPDF lines + mark downstream source as pymupdf for these pages
        if page_num in pymupdf_text_pages_set:
            try:
                pm_page = pm_doc.load_page(i)
                page_obj["pymupdf"]["lines"] = pymupdf_lines_with_coords(pm_page)
                page_obj["stage1"]["text_source_for_downstream"] = "pymupdf"
            except Exception as e:
                page_warn.append({"type": "pymupdf_lines_error", "page_num": page_num, "error": str(e)})
                page_obj["pymupdf"]["lines"] = None

        if page_warn:
            warnings.extend(page_warn)

        # Summary metrics
        pl_lines_text = "\n".join([ln.get("text", "") for ln in page_obj["pdfplumber"]["lines"]])
        pm_lines_len = 0
        if isinstance(page_obj["pymupdf"].get("lines"), list):
            pm_lines_len = len("\n".join([ln.get("text", "") for ln in page_obj["pymupdf"]["lines"]]))

        summary_rows.append({
            "page_num": page_num,
            "width": page_obj["width"],
            "height": page_obj["height"],
            "num_words_pdfplumber": len(page_obj["pdfplumber"]["words"]),
            "num_lines_pdfplumber": len(page_obj["pdfplumber"]["lines"]),
            "text_len_pdfplumber": len(_safe_text(pl_lines_text)),
            "text_len_pymupdf": len(page_obj["pymupdf"]["text"] or ""),
            "num_lines_pymupdf": (len(page_obj["pymupdf"]["lines"]) if isinstance(page_obj["pymupdf"].get("lines"), list) else 0),
            "text_len_pymupdf_lines": int(pm_lines_len),
            "text_source_for_downstream": page_obj["stage1"]["text_source_for_downstream"],
        })

        # Persist
        fjsonl.write(json.dumps(page_obj, ensure_ascii=False) + "\n")
        ctx["pages"].append(page_obj)

pl_doc.close()
pm_doc.close()

print("Saved:", PAGES_JSONL_PATH)
print("Pages in ctx['pages']:", len(ctx["pages"]))

Saved: ../../data/processed/book1/pages_raw.jsonl
Pages in ctx['pages']: 81


In [9]:
# STAGE 1 — Save pages_summary.csv + stage1_report.json

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(PAGES_SUMMARY_CSV_PATH, index=False)

total_pages = int(len(df_summary))
avg_words = float(df_summary["num_words_pdfplumber"].mean()) if total_pages else 0.0
avg_lines = float(df_summary["num_lines_pdfplumber"].mean()) if total_pages else 0.0

stage1_report = {
    "total_pages": total_pages,
    "avg_words_per_page": avg_words,
    "avg_lines_per_page": avg_lines,
    "pymupdf_text_pages_configured": list(PYMUPDF_TEXT_PAGES),
    "warnings_count": int(len(ctx["reports"]["stage1_warnings"])),
    "errors_count": int(len(ctx["reports"]["stage1_errors"])),
    "warnings_sample": ctx["reports"]["stage1_warnings"][:20],
}

with open(STAGE1_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(stage1_report, f, ensure_ascii=False, indent=2)

print("Saved:", PAGES_SUMMARY_CSV_PATH)
print("Saved:", STAGE1_REPORT_PATH)

display(df_summary.head(10))

Saved: ../../data/processed/book1/pages_summary.csv
Saved: ../../data/processed/book1/stage1_report.json


,page_num,width,height,num_words_pdfplumber,num_lines_pdfplumber,text_len_pdfplumber,text_len_pymupdf,num_lines_pymupdf,text_len_pymupdf_lines,text_source_for_downstream
0,1,595.44,841.68,71,12,478,487,0,0,pdfplumber
1,2,595.44,841.68,281,26,1949,1954,0,0,pdfplumber
2,3,595.44,841.68,213,27,1574,1580,0,0,pdfplumber
3,4,595.44,841.68,209,40,1321,1331,0,0,pdfplumber
4,5,595.44,841.68,275,40,1688,1698,0,0,pdfplumber
5,6,595.44,841.68,96,15,618,630,0,0,pdfplumber
6,7,595.44,841.68,195,25,1179,1201,0,0,pdfplumber
7,8,595.44,841.68,115,16,722,736,0,0,pdfplumber
8,9,595.44,841.68,67,10,431,437,0,0,pdfplumber
9,10,595.44,841.68,292,36,1969,1969,0,0,pdfplumber


In [10]:
# STAGE 1 — Debug helper + validation

def debug_page(page_num: int, max_lines: int = 20, max_words: int = 30):
    if not ctx.get("pages"):
        print("ctx['pages'] is empty.")
        return
    if page_num < 1 or page_num > len(ctx["pages"]):
        print(f"Invalid page_num {page_num}. Must be 1..{len(ctx['pages'])}")
        return

    p = ctx["pages"][page_num - 1]

    pl_lines = p.get("pdfplumber", {}).get("lines", []) or []
    pl_words = p.get("pdfplumber", {}).get("words", []) or []
    pm_text = p.get("pymupdf", {}).get("text", "") or ""
    pm_lines = p.get("pymupdf", {}).get("lines", None)

    source = (p.get("stage1", {}) or {}).get("text_source_for_downstream", "pdfplumber")

    print(f"=== PAGE {page_num} | downstream_text_source={source} ===")
    print(f"Size: width={p.get('width')}, height={p.get('height')}")
    print(f"pdfplumber: {len(pl_lines)} lines, {len(pl_words)} words")
    print(f"pymupdf: text_len={len(pm_text)} | lines={'None' if pm_lines is None else len(pm_lines)}")

    print("\n--- First lines (pdfplumber-derived) ---")
    for i, ln in enumerate(pl_lines[:max_lines]):
        print(f"{i+1:02d} top={ln.get('top'):.2f} x0={ln.get('x0'):.2f} x1={ln.get('x1'):.2f} :: {ln.get('text','')[:160]}")

    if isinstance(pm_lines, list) and pm_lines:
        print("\n--- First lines (PyMuPDF dict-derived) ---")
        for i, ln in enumerate(pm_lines[:max_lines]):
            print(f"{i+1:02d} top={ln.get('top'):.2f} x0={ln.get('x0'):.2f} x1={ln.get('x1'):.2f} :: {ln.get('text','')[:160]}")

    print("\n--- First words (pdfplumber) ---")
    for i, w in enumerate(pl_words[:max_words]):
        print(f"{i+1:02d} top={w.get('top'):.2f} x0={w.get('x0'):.2f} x1={w.get('x1'):.2f} :: {w.get('text','')[:80]}")

    print("\n--- PyMuPDF text preview ---")
    print(pm_text[:8000])

print("Total pages extracted:", len(ctx["pages"]))
print("Avg words/page:", float(df_summary["num_words_pdfplumber"].mean()))
print("Avg lines/page:", float(df_summary["num_lines_pdfplumber"].mean()))
print("Warnings:", len(ctx["reports"]["stage1_warnings"]))

# Debug page 1 and a middle page
debug_page(1)
mid = max(1, len(ctx["pages"]) // 2)
debug_page(mid)

# Debug first configured PyMuPDF text page (if any)


Total pages extracted: 81
Avg words/page: 264.77777777777777
Avg lines/page: 36.51851851851852
Warnings: 0
=== PAGE 1 | downstream_text_source=pdfplumber ===
Size: width=595.44, height=841.68
pdfplumber: 12 lines, 71 words
pymupdf: text_len=487 | lines=None

--- First lines (pdfplumber-derived) ---
01 top=743.66 x0=72.00 x1=523.49 :: © 1994-2022 EMVCo, LLC (“EMVCo”). All rights reserved. Reproduction, distribution and other use of
02 top=755.12 x0=72.00 x1=523.51 :: this document is permitted only pursuant to the applicable agreement between the user and EMVCo
03 top=766.64 x0=71.99 x1=523.44 :: found at www.emvco.com. EMV® is a registered trademark or trademark of EMVCo, LLC in the United
04 top=778.16 x0=72.00 x1=191.56 :: States and other countries.
05 top=76.76 x0=72.00 x1=140.85 :: EMV ®
06 top=109.40 x0=72.00 x1=332.05 :: Integrated Circuit Card
07 top=137.00 x0=72.00 x1=485.37 :: Specifications for Payment Systems
08 top=199.97 x0=72.00 x1=158.63 :: Book 1
09 top=244.09 x0=72.00

In [11]:
if PYMUPDF_TEXT_PAGES:
    debug_page(PYMUPDF_TEXT_PAGES[0])

=== PAGE 12 | downstream_text_source=pymupdf ===
Size: width=595.44, height=841.68
pdfplumber: 43 lines, 272 words
pymupdf: text_len=2012 | lines=55

--- First lines (pdfplumber-derived) ---
01 top=38.66 x0=72.00 x1=144.29 :: EMV 4.4 Book 1
02 top=50.12 x0=71.99 x1=210.47 :: Application Independent ICC to
03 top=61.64 x0=71.99 x1=217.65 :: Terminal Interface Requirements
04 top=732.14 x0=72.00 x1=523.49 :: October 2022 Page 12
05 top=755.12 x0=72.00 x1=523.49 :: © 1994-2022 EMVCo, LLC (“EMVCo”). All rights reserved. Reproduction, distribution and other use of
06 top=766.64 x0=72.00 x1=523.52 :: this document is permitted only pursuant to the applicable agreement between the user and EMVCo
07 top=778.17 x0=72.00 x1=523.43 :: found at www.emvco.com. EMV® is a registered trademark or trademark of EMVCo, LLC in the United
08 top=789.62 x0=72.00 x1=191.56 :: States and other countries.
09 top=125.63 x0=72.00 x1=298.10 :: 2 Normative References
10 top=161.95 x0=72.00 x1=497.88 :: The followi

In [12]:
display(df_summary.iloc[22])

page_num                           23
width                          595.44
height                         841.68
num_words_pdfplumber              167
num_lines_pdfplumber               33
text_len_pdfplumber              1253
text_len_pymupdf                 1278
num_lines_pymupdf                  57
text_len_pymupdf_lines           1252
text_source_for_downstream    pymupdf
Name: 22, dtype: object